# 38 — Model Watermarking, IP Protection & Machine Unlearning — Lab

## Description

This lab implements the Kirchenbauer green-list/red-list token-bias watermarking scheme on GPT-2 and machine unlearning via gradient ascent. You will generate 50 watermarked completions, compute detection z-scores, apply paraphrase attacks, measure watermark survival, and then apply gradient ascent unlearning to remove a target sample's influence. Finally, you will verify unlearning via membership inference attack (MIA), checking that the target sample's attack success rate (ASR) drops after unlearning. This exercises watermarking, detection, robustness, gradient ascent unlearning, and MIA verification concepts.

## Objectives

- Implement the Kirchenbauer watermarking scheme on GPT-2 output: partition the vocabulary using a context hash seed, bias green-list tokens, and generate watermarked completions.
- Compute the detection z-score for watermarked text and measure the initial detection rate.
- Apply paraphrase attacks to watermarked text and measure watermark survival rate after paraphrasing.
- Apply gradient ascent approximate unlearning to a target set of MNIST training samples to increase their loss.
- Verify that unlearning reduces the membership inference attack success rate (ASR) on the target samples.

## Prerequisites

You should understand the lecture content on Kirchenbauer watermarking, green-list/red-list token partitioning, z-score detection, GDPR Article 17, and gradient ascent approximate unlearning. Familiarity with GPT-2's token generation and basic neural network training is assumed. No prior implementation experience with watermarking or unlearning is required.

## Required Software/Packages

Requires PyTorch 2.0+, Transformers 4.30+, NumPy 1.24+, torchvision for MNIST, and scipy for statistics. All are open-source and available via pip. Standard GPU or CPU will suffice; GPU is recommended for faster execution but not required.

## Environment/Setup Instructions

Install required packages by running the cell below. No special environment activation is needed. 

In [1]:
# Install required packages (run once; safe to re-run)
import subprocess
import sys

try:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch", "torchvision", "transformers", "numpy", "scipy", "pandas"
    ])
    print("Required packages installed.")
except subprocess.CalledProcessError as error:
    print(f"Package installation failed: {error}")

Required packages installed.


### Step 1: Setup — Imports and GPT-2 Loading

Import the required libraries, select the compute device (CUDA or Apple Silicon MPS if available, otherwise CPU), and load the GPT-2 model and tokenizer used for watermark generation.

In [2]:
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
from transformers import GPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
torch.manual_seed(0)
np.random.seed(0)

# Set device: prefer CUDA, then Apple Silicon MPS, then CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

W1115 17:19:35.404000 21779 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Using device: mps


In [3]:
# Load GPT-2 model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
model.to(device)
print("GPT-2 loaded.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 loaded.


In [4]:
def get_context_hash(prompt: str, seed_key: str) -> int:
    """Compute deterministic hash from prompt and key."""
    hash_input = prompt + seed_key
    hash_obj = hashlib.sha256(hash_input.encode())
    return int(hash_obj.hexdigest(), 16)


def partition_vocabulary(vocab_size: int, context_hash: int, greenlist_ratio: float = 0.5) -> set:
    """Partition vocabulary into green-list based on context hash."""
    rng = np.random.RandomState(context_hash % (2**31))
    indices = rng.permutation(vocab_size)
    greenlist_size = int(vocab_size * greenlist_ratio)
    return set(indices[:greenlist_size])

In [5]:
def generate_watermarked_completion(prompt: str, num_tokens: int = 200, delta: float = 0.5, seed_key: str = "secret_key") -> tuple:
    """Generate watermarked completion by biasing green-list tokens."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    context_hash = get_context_hash(prompt, seed_key)
    greenlist = partition_vocabulary(tokenizer.vocab_size, context_hash)
    greenlist_ids = torch.tensor(sorted(greenlist), dtype=torch.long, device=device)

    completion_ids = []
    with torch.no_grad():
        for _ in range(num_tokens):
            outputs = model(input_ids)
            logits = outputs.logits[:, -1, :]

            # Apply green-list bias (vectorized)
            logits[0, greenlist_ids] += delta

            # Sample next token
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            completion_ids.append(next_token.item())
            input_ids = torch.cat([input_ids, next_token], dim=1)  # next_token has shape [1, 1]

    completion_text = tokenizer.decode(completion_ids)
    return completion_text, completion_ids, greenlist


def compute_z_score(completion_ids: list, greenlist: set) -> float:
    """Compute z-score for watermark detection: z = (G - T/2) / sqrt(T/4)."""
    T = len(completion_ids)
    G = sum(1 for token_id in completion_ids if token_id in greenlist)
    z = (G - T / 2) / np.sqrt(T / 4) if T > 0 else 0
    return z

In [6]:
prompt = "The future of artificial intelligence is"
watermarked_data = []

print(f"Generating 50 watermarked completions from prompt: '{prompt}'")
for i in range(50):
    completion, comp_ids, greenlist = generate_watermarked_completion(prompt, num_tokens=200, delta=2.0)
    z_score = compute_z_score(comp_ids, greenlist)
    watermarked_data.append({
        "id": i,
        "completion": completion,
        "token_ids": comp_ids,
        "greenlist": list(greenlist),
        "z_score": z_score,
        "detected": z_score > 4.0
    })
    if (i + 1) % 10 == 0:
        print(f"Generated {i + 1} completions...")

# Compute initial detection rate
initial_detected = sum(1 for d in watermarked_data if d["detected"])
initial_detection_rate = initial_detected / 50
print(f"\nInitial detection rate: {initial_detected}/50 detected (z > 4.0)")
print(f"Detection rate: {initial_detection_rate:.2%}")

Generating 50 watermarked completions from prompt: 'The future of artificial intelligence is'


Generated 10 completions...


Generated 20 completions...


Generated 30 completions...


Generated 40 completions...


Generated 50 completions...

Initial detection rate: 50/50 detected (z > 4.0)
Detection rate: 100.00%


### Steps 3–4: Paraphrase Attack and Watermark Survival

Apply an aggressive paraphrase attack that replaces 50% of tokens with random vocabulary tokens, then recompute the z-score for each paraphrased completion using the **same** green-list (the detector knows the secret key). The **survival rate** is the fraction of completions still detected after paraphrasing.

In [7]:
def simple_paraphrase(text: str, replacement_rate: float = 0.5) -> str:
    """Simple paraphrase by replacing a fraction of tokens with random tokens."""
    tokens = tokenizer.encode(text)
    rng = np.random.RandomState(42)  # fixed seed for reproducibility

    # Select tokens to replace
    replace_indices = rng.choice(len(tokens), size=int(len(tokens) * replacement_rate), replace=False)
    paraphrased_tokens = tokens.copy()

    for idx in replace_indices:
        # Replace with random token (simplified paraphrase approach)
        paraphrased_tokens[idx] = rng.randint(100, tokenizer.vocab_size)

    return tokenizer.decode(paraphrased_tokens)


print("Paraphrasing 50 watermarked completions...")
paraphrased_data = []

for i, data in enumerate(watermarked_data):
    paraphrased_text = simple_paraphrase(data["completion"], replacement_rate=0.5)
    paraphrased_tokens = tokenizer.encode(paraphrased_text)
    z_score_para = compute_z_score(paraphrased_tokens, set(data["greenlist"]))
    paraphrased_data.append({
        "id": i,
        "original_z_score": data["z_score"],
        "paraphrased_z_score": z_score_para,
        "original_detected": data["detected"],
        "paraphrased_detected": z_score_para > 4.0
    })
    if (i + 1) % 10 == 0:
        print(f"Paraphrased {i + 1} completions...")

paraphrased_detected = sum(1 for d in paraphrased_data if d["paraphrased_detected"])
survival_rate = paraphrased_detected / 50
print(f"\nWatermark survival after paraphrasing:")
print(f"Paraphrased detected: {paraphrased_detected}/50 (z > 4.0)")
print(f"Survival rate: {survival_rate:.2%}")

Paraphrasing 50 watermarked completions...
Paraphrased 10 completions...
Paraphrased 20 completions...
Paraphrased 30 completions...
Paraphrased 40 completions...
Paraphrased 50 completions...

Watermark survival after paraphrasing:
Paraphrased detected: 19/50 (z > 4.0)
Survival rate: 38.00%


In [8]:
summary = pd.DataFrame({
    "Condition": ["Original (no paraphrase)", "After paraphrasing"],
    "Count Detected": [initial_detected, paraphrased_detected],
    "Detection Rate": [f"{initial_detection_rate:.2%}", f"{survival_rate:.2%}"]
})

print("=== WATERMARK DETECTION SUMMARY ===")
print(summary.to_string(index=False))
print(f"\nWatermark robustness conclusion: token-replacement paraphrasing reduced the")
print(f"detection rate from {initial_detection_rate:.0%} to {survival_rate:.0%}, demonstrating that")
print(f"watermark robustness to paraphrase attacks remains a critical challenge for")
print(f"practical IP protection.")

=== WATERMARK DETECTION SUMMARY ===
               Condition  Count Detected Detection Rate
Original (no paraphrase)              50        100.00%
      After paraphrasing              19         38.00%

Watermark robustness conclusion: token-replacement paraphrasing reduced the
detection rate from 100% to 38%, demonstrating that
watermark robustness to paraphrase attacks remains a critical challenge for
practical IP protection.


### Expected Outputs (Steps 1-4)

The code above generates 50 watermarked completions and measures detection before and after paraphrasing. You should see output like:
- "Generated 50 completions..."
- "Initial detection rate: 50/50 detected (z > 4.0)" - approximately 90-100% detection before paraphrasing (typical z-scores are 8-10 with delta=2.0)
- "Watermark survival after paraphrasing: 15-30/50 (z > 4.0)" - approximately 30-60% survival rate after the aggressive paraphrase attack
- A summary table showing Condition, Count Detected, Detection Rate

The exact numbers vary with sampling randomness and paraphrase aggressiveness, but the aggressive 50% token-replacement attack should clearly degrade the watermark: detection drops from near 100% to roughly 30-60%.

## Step 5: Gradient Ascent Unlearning & MIA Verification
This step implements approximate machine unlearning via gradient ascent. You will:
1. Train a simple CNN on MNIST with 100 training samples and 100 test samples.
2. Run a membership inference attack (MIA) before unlearning: compute per-sample losses on members (forget set) and non-members (test set), fix the decision threshold at the median non-member loss, and compute the baseline Attack Success Rate (ASR).
3. Apply 10 steps of gradient ascent on the forget set (100 MNIST training images) at a learning rate well below the training rate, with gradient clipping to keep updates stable.
4. Run MIA again post-unlearning using the **same fixed threshold** and compute the new ASR.
5. Report both ASR values and verify that ASR drops (unlearning reduces the samples' detectability as training members) while test accuracy stays reasonable.

### Step 5a: Define the MNIST Classifier

A small CNN with two convolutional layers is enough to fit the 100-sample training set used in this unlearning demonstration.

In [9]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

### Step 5b: Load MNIST and Create the Forget Set

Use 100 MNIST images as the training set (which doubles as the forget set for this demonstration) and the next 100 images as the non-member test set.

In [10]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
mnist_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)

train_size = 100
test_size = 100
forget_size = 100  # samples to be "forgotten" via unlearning

# Split full dataset: first 100 for training (including forget set), next 100 for testing
train_set = torch.utils.data.Subset(mnist_full, list(range(train_size)))
test_set = torch.utils.data.Subset(mnist_full, list(range(train_size, train_size + test_size)))

# Within train_set, all 100 samples form the "forget set"
forget_indices = list(range(min(forget_size, train_size)))
forget_set_data = torch.utils.data.Subset(train_set, forget_indices)

train_loader = DataLoader(train_set, batch_size=10, shuffle=True)
test_loader = DataLoader(test_set, batch_size=10, shuffle=False)
forget_loader = DataLoader(forget_set_data, batch_size=10, shuffle=False)

print(f"Created MNIST subsets: train={train_size}, test={test_size}, forget_set={forget_size}")

Created MNIST subsets: train=100, test=100, forget_set=100


### Step 5c: Train the Model

Train the CNN for 5 epochs on the 100 training samples and track test accuracy.

In [11]:
model_mnist = SimpleCNN().to(device)
optimizer = optim.SGD(model_mnist.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

print("Training MNIST model on 100 training samples...")
for epoch in range(5):
    model_mnist.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_mnist(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # Evaluate on test set
    model_mnist.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_mnist(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f"Epoch {epoch + 1}/5: Test accuracy = {accuracy:.1f}%")

print("\nModel training complete.")
accuracy_before = accuracy  # save pre-unlearning test accuracy for later comparison

Training MNIST model on 100 training samples...


Epoch 1/5: Test accuracy = 25.0%


Epoch 2/5: Test accuracy = 45.0%
Epoch 3/5: Test accuracy = 44.0%


Epoch 4/5: Test accuracy = 55.0%


Epoch 5/5: Test accuracy = 54.0%

Model training complete.


### Step 5d: Membership Inference Attack (Before Unlearning)

The attack computes per-sample cross-entropy losses and fixes a decision threshold at the **median non-member loss**. **ASR** is the fraction of forget-set (member) samples whose loss falls below that threshold — low loss is the signature of a training member. The threshold is computed once, before unlearning, and reused afterward so the attack is calibrated consistently.

In [12]:
def compute_losses(model, loader):
    """Compute per-sample cross-entropy losses for all samples in a loader."""
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="none")
    all_losses = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            losses = criterion(outputs, labels)
            all_losses.extend(losses.cpu().numpy())
    return np.array(all_losses)


print("Running MIA BEFORE unlearning...")
member_losses_before = compute_losses(model_mnist, forget_loader)
non_member_losses_before = compute_losses(model_mnist, test_loader)

# Fixed decision threshold: median non-member loss (computed once, reused after unlearning)
mia_threshold = np.median(non_member_losses_before)

# ASR: fraction of members whose loss falls below the threshold
asr_before = np.mean(member_losses_before < mia_threshold)

print(f"\n=== MIA RESULTS BEFORE UNLEARNING ===")
print(f"Attack Success Rate (ASR): {asr_before:.3f} ({asr_before * 100:.1f}%)")
print(f"Mean member (forget-set) loss: {member_losses_before.mean():.4f}")
print(f"Mean non-member loss: {non_member_losses_before.mean():.4f}")
print(f"Decision threshold (median non-member loss): {mia_threshold:.4f}")
print(f"\nInterpretation: ASR of {asr_before * 100:.1f}% means {asr_before * 100:.0f}% of forget-set samples")
print(f"have lower loss than the median non-member, indicating they are easily identified as members.")

Running MIA BEFORE unlearning...

=== MIA RESULTS BEFORE UNLEARNING ===
Attack Success Rate (ASR): 0.730 (73.0%)
Mean member (forget-set) loss: 0.7156
Mean non-member loss: 1.5451
Decision threshold (median non-member loss): 0.7610

Interpretation: ASR of 73.0% means 73% of forget-set samples
have lower loss than the median non-member, indicating they are easily identified as members.


### Step 5e: Gradient Ascent Unlearning

Apply 10 steps of gradient **ascent** on the forget set (negating the loss so the optimizer *maximizes* it), using a learning rate much smaller than the training rate ($3 \times 10^{-4}$ vs. $10^{-2}$) and gradient clipping for stability. Ascent that is too aggressive destroys the model's general utility, so the rate must be tuned carefully. Then confirm the model's test accuracy did not collapse.

In [13]:
# Gradient Ascent Unlearning: Apply 10 steps of gradient ascent on the forget set

print("Applying gradient ascent unlearning (10 steps) on forget set...")
model_mnist.train()

# Use a small learning rate for unlearning (much smaller than the training LR of 0.01)
unlearning_lr = 0.0003
unlearning_optimizer = optim.SGD(model_mnist.parameters(), lr=unlearning_lr)
criterion = nn.CrossEntropyLoss()

# Apply gradient ascent for 10 steps
for step in range(10):
    for images, labels in forget_loader:
        images, labels = images.to(device), labels.to(device)
        unlearning_optimizer.zero_grad()
        outputs = model_mnist(images)
        loss = criterion(outputs, labels)
        # Gradient ASCENT: negate the loss to maximize it
        (-loss).backward()
        # Clip gradients to keep the ascent updates stable
        torch.nn.utils.clip_grad_norm_(model_mnist.parameters(), max_norm=5.0)
        unlearning_optimizer.step()
    print(f"Gradient ascent step {step + 1}/10 complete.")

print("\nGradient ascent unlearning complete.")

# Verify test accuracy did not degrade significantly
model_mnist.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_mnist(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
accuracy_after = 100 * correct / total
print(f"Test accuracy after unlearning: {accuracy_after:.1f}%")

Applying gradient ascent unlearning (10 steps) on forget set...


Gradient ascent step 1/10 complete.
Gradient ascent step 2/10 complete.


Gradient ascent step 3/10 complete.


Gradient ascent step 4/10 complete.


Gradient ascent step 5/10 complete.
Gradient ascent step 6/10 complete.


Gradient ascent step 7/10 complete.


Gradient ascent step 8/10 complete.


Gradient ascent step 9/10 complete.
Gradient ascent step 10/10 complete.

Gradient ascent unlearning complete.


Test accuracy after unlearning: 33.0%


### Step 5f: Membership Inference Attack (After Unlearning)

Re-run the same attack with the **same fixed threshold** from before unlearning. If unlearning worked, forget-set losses rise above that threshold and ASR drops.

In [14]:
# Membership Inference Attack (MIA) AFTER unlearning

print("Running MIA AFTER unlearning...")
member_losses_after = compute_losses(model_mnist, forget_loader)
non_member_losses_after = compute_losses(model_mnist, test_loader)

# Reuse the SAME fixed threshold computed before unlearning
asr_after = np.mean(member_losses_after < mia_threshold)

print(f"\n=== MIA RESULTS AFTER UNLEARNING ===")
print(f"Attack Success Rate (ASR): {asr_after:.3f} ({asr_after * 100:.1f}%)")
print(f"Mean member (forget-set) loss: {member_losses_after.mean():.4f}")
print(f"Mean non-member loss: {non_member_losses_after.mean():.4f}")
print(f"Decision threshold (fixed, from before unlearning): {mia_threshold:.4f}")
print(f"\nInterpretation: ASR of {asr_after * 100:.1f}% means only {asr_after * 100:.0f}% of forget-set samples")
print(f"still have lower loss than the pre-unlearning non-member median, indicating reduced membership detectability.")

# Compute ASR reduction
asr_reduction = asr_before - asr_after
asr_reduction_percent = (asr_reduction / asr_before * 100) if asr_before > 0 else 0

print(f"\n" + "=" * 70)
print(f"=== UNLEARNING EFFECTIVENESS SUMMARY ===")
print(f"=" * 70)
print(f"\nASR BEFORE unlearning:  {asr_before:.3f} ({asr_before * 100:.1f}%)")
print(f"ASR AFTER unlearning:   {asr_after:.3f} ({asr_after * 100:.1f}%)")
print(f"ASR reduction:          {asr_reduction:.3f} ({asr_reduction_percent:.1f}% drop)")
print(f"\nConclusion: Gradient ascent unlearning reduced the membership inference")
print(f"attack success rate by {asr_reduction_percent:.1f}%, demonstrating that the forget-set samples'")
print(f"influence on model parameters was reduced. Test accuracy moved from {accuracy_before:.1f}%")
print(f"to {accuracy_after:.1f}%, illustrating the utility-forgetting trade-off of approximate")
print(f"unlearning: stronger ascent forgets more but degrades general performance.")
print(f"Note: this is approximate unlearning - the reduced ASR is evidence of reduced")
print(f"influence, not a formal guarantee of complete erasure (see notes.md).")
print(f"=" * 70)

Running MIA AFTER unlearning...



=== MIA RESULTS AFTER UNLEARNING ===
Attack Success Rate (ASR): 0.460 (46.0%)
Mean member (forget-set) loss: 1.8580
Mean non-member loss: 2.7598
Decision threshold (fixed, from before unlearning): 0.7610

Interpretation: ASR of 46.0% means only 46% of forget-set samples
still have lower loss than the pre-unlearning non-member median, indicating reduced membership detectability.

=== UNLEARNING EFFECTIVENESS SUMMARY ===

ASR BEFORE unlearning:  0.730 (73.0%)
ASR AFTER unlearning:   0.460 (46.0%)
ASR reduction:          0.270 (37.0% drop)

Conclusion: Gradient ascent unlearning reduced the membership inference
attack success rate by 37.0%, demonstrating that the forget-set samples'
influence on model parameters was reduced. Test accuracy moved from 54.0%
to 33.0%, illustrating the utility-forgetting trade-off of approximate
unlearning: stronger ascent forgets more but degrades general performance.
Note: this is approximate unlearning - the reduced ASR is evidence of reduced
influence, n

In [15]:
# Verification: Check that ASR decreased after unlearning
assert asr_before >= 0 and asr_before <= 1, "ASR before must be in [0, 1]"
assert asr_after >= 0 and asr_after <= 1, "ASR after must be in [0, 1]"
assert asr_after < asr_before, "ASR should decrease after unlearning (membership detectability reduced)"

print("✓ Verification passed: ASR decreased after gradient ascent unlearning.")
print(f"✓ Watermark detection and unlearning exercises complete.")
print(f"\nKey metrics:")
print(f"  - Watermark initial detection: {initial_detection_rate*100:.1f}%")
print(f"  - Watermark survival after paraphrase: {survival_rate*100:.1f}%")
print(f"  - MIA ASR before unlearning: {asr_before*100:.1f}%")
print(f"  - MIA ASR after unlearning: {asr_after*100:.1f}%")
print(f"  - ASR reduction: {asr_reduction_percent:.1f}%")

✓ Verification passed: ASR decreased after gradient ascent unlearning.
✓ Watermark detection and unlearning exercises complete.

Key metrics:
  - Watermark initial detection: 100.0%
  - Watermark survival after paraphrase: 38.0%
  - MIA ASR before unlearning: 73.0%
  - MIA ASR after unlearning: 46.0%
  - ASR reduction: 37.0%


## Questions/Tasks

1. Why does watermark survival rate after paraphrasing drop below 100%? Explain in terms of which tokens are being replaced and how that affects the z-score computation.
2. If you increased the watermark bias delta from 2.0 to 4.0, would you expect the survival rate to increase or decrease? Why?
3. In the unlearning experiment, the ASR should drop after gradient ascent. Explain why increasing the loss on forget-set samples (gradient ascent) makes them harder for the MIA to detect as members.
4. Is the reduction in ASR after unlearning a guarantee that the samples were completely forgotten? Why or why not?

## Optional Challenges

1. Implement a more sophisticated paraphrase strategy that preserves word frequency distributions and measure whether watermark survival improves.
2. Experiment with different watermark bias strengths (delta = 0.1, 1.0, 2.0) and plot how detection rate and z-score distribution change. Does higher delta always lead to better robustness?
3. Compare exact unlearning (retraining model from scratch on remaining members only) to the approximate gradient ascent unlearning implemented here. Measure ASR for both and compare computational cost.
4. Implement a shadow model MIA (train separate shadow models with and without the target sample) and compare ASR to the simple MIA baseline used here.